# Buổi 7: Học máy cơ bản và mô hình hồi quy
**Môn học:** Python cho Phân tích Dữ liệu Kinh doanh · **BÀI TẬP THỰC HÀNH**

**Mục tiêu:**
- Nắm quy trình học máy với scikit-learn: chuẩn bị X, y → chia train/test → fit → predict → đánh giá
- Xây dựng mô hình Linear Regression và Random Forest Regressor
- Đánh giá mô hình bằng MAE, RMSE, R² và diễn giải hệ số cho quyết định kinh doanh
- Xử lý biến phân loại bằng one-hot encoding

**Dữ liệu dùng trong buổi:** `advertising.csv`, `house_prices.csv` (trong thư mục `data/`). Nếu dùng Google Colab, tải thư mục `data/` lên rồi sửa biến `DATA` bên dưới.

**Cấu trúc:** Phần A – bài tập có hướng dẫn (làm theo, chạy từng ô) · Phần B – bài tập tự làm (bắt buộc, nộp cuối buổi) · Phần C – bài tập làm thêm (dành cho bạn muốn thử thách).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
DATA = "../data/"
sns.set_theme(style="whitegrid")

**Quy trình 5 bước dùng cho mọi mô hình trong scikit-learn:**
1. Chuẩn bị `X` (các biến đầu vào) và `y` (biến cần dự báo)
2. Chia dữ liệu: `train_test_split` (thường 80% huấn luyện, 20% kiểm tra)
3. Tạo mô hình và huấn luyện: `model.fit(X_train, y_train)`
4. Dự báo: `model.predict(X_test)`
5. Đánh giá trên tập kiểm tra (dữ liệu mô hình chưa thấy)

## Phần A. Bài tập có hướng dẫn

**Bài toán:** Công ty có dữ liệu 200 thị trường: chi phí quảng cáo TV, radio, mạng xã hội (triệu đồng) và doanh số (nghìn sản phẩm). Cần dự báo doanh số và quyết định phân bổ ngân sách.

### A1. Khám phá dữ liệu

In [ ]:
ad = pd.read_csv(DATA + "advertising.csv")
print(ad.shape); print(ad.describe().round(1))
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), sharey=True)
for ax, c in zip(axes, ["tv", "radio", "mang_xa_hoi"]):
    sns.regplot(data=ad, x=c, y="doanh_so", ax=ax, scatter_kws={"s": 12, "alpha": 0.6})
plt.tight_layout(); plt.show()
print(ad.corr()["doanh_so"].round(2))

### A2. Hồi quy tuyến tính đơn: doanh số theo chi phí TV

In [ ]:
X = ad[["tv"]]            # X phải là bảng 2 chiều (DataFrame)
y = ad["doanh_so"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(len(X_train), "dòng huấn luyện,", len(X_test), "dòng kiểm tra")

lr1 = LinearRegression()
lr1.fit(X_train, y_train)
print(f"doanh_so = {lr1.intercept_:.2f} + {lr1.coef_[0]:.4f} × tv")

**Diễn giải:** mỗi 1 triệu đồng chi thêm cho TV gắn với doanh số tăng thêm khoảng `coef` nghìn sản phẩm (khi các yếu tố khác không đổi).
Hệ số chặn là doanh số dự kiến khi không quảng cáo TV.

### A3. Đánh giá mô hình
- **MAE** – sai số tuyệt đối trung bình (cùng đơn vị với y, dễ giải thích)
- **RMSE** – căn của sai số bình phương trung bình (phạt nặng sai số lớn)
- **R²** – tỷ lệ biến động của y được mô hình giải thích (0 → 1, càng gần 1 càng tốt)

In [ ]:
def danh_gia(model, X_te, y_te, ten=""):
    p = model.predict(X_te)
    kq = {"Mô hình": ten, "MAE": mean_absolute_error(y_te, p),
          "RMSE": mean_squared_error(y_te, p) ** 0.5, "R²": r2_score(y_te, p)}
    return {k: (round(v, 3) if isinstance(v, float) else v) for k, v in kq.items()}

print(danh_gia(lr1, X_test, y_test, "LR – chỉ TV"))

### A4. Hồi quy tuyến tính bội: dùng cả 3 kênh

In [ ]:
X = ad[["tv", "radio", "mang_xa_hoi"]]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
lr = LinearRegression().fit(X_train, y_train)
he_so = pd.Series(lr.coef_, index=X.columns).round(4)
print("Hệ số chặn:", round(lr.intercept_, 3)); print(he_so)
print(danh_gia(lr, X_test, y_test, "LR – 3 kênh"))

In [ ]:
# Biểu đồ thực tế – dự báo: điểm càng sát đường chéo, mô hình càng tốt
p = lr.predict(X_test)
plt.figure(figsize=(5, 5))
plt.scatter(y_test, p, alpha=0.7)
lim = [y.min(), y.max()]; plt.plot(lim, lim, "r--")
plt.xlabel("Doanh số thực tế"); plt.ylabel("Doanh số dự báo"); plt.title("Thực tế và dự báo"); plt.show()

**Ý nghĩa kinh doanh:** so sánh hệ số → 1 triệu đồng chi cho kênh nào tạo thêm nhiều doanh số nhất.
Hệ số của mạng xã hội nhỏ cho thấy kênh này (trong dữ liệu này) kém hiệu quả hơn radio.

### A5. Random Forest Regressor và so sánh mô hình

In [ ]:
rf = RandomForestRegressor(n_estimators=300, random_state=42)
rf.fit(X_train, y_train)
bang = pd.DataFrame([danh_gia(lr1, X_test[["tv"]], y_test, "LR – chỉ TV"),
                     danh_gia(lr, X_test, y_test, "LR – 3 kênh"),
                     danh_gia(rf, X_test, y_test, "Random Forest – 3 kênh")])
bang

In [ ]:
pd.Series(rf.feature_importances_, index=X.columns).sort_values().plot(kind="barh", title="Mức độ quan trọng của biến (Random Forest)")
plt.show()

### A6. Dùng mô hình để ra quyết định

In [ ]:
phuong_an = pd.DataFrame({
    "tv":          [200, 150, 120],
    "radio":       [20,  45,  30],
    "mang_xa_hoi": [30,  55,  100],
}, index=["PA1: dồn TV", "PA2: TV + radio", "PA3: dồn MXH"])
phuong_an["tong_ngan_sach"] = phuong_an.sum(axis=1)
phuong_an["du_bao_LR"] = lr.predict(phuong_an[["tv", "radio", "mang_xa_hoi"]]).round(2)
phuong_an["du_bao_RF"] = rf.predict(phuong_an[["tv", "radio", "mang_xa_hoi"]]).round(2)
phuong_an

Cùng ngân sách 250 triệu đồng, phương án nào cho doanh số dự báo cao nhất? Lưu ý: mô hình chỉ đáng tin trong phạm vi dữ liệu đã học.

## Phần B. Bài tập tự làm

**Bài toán:** Dự báo giá căn hộ tại TP.HCM từ `house_prices.csv` (giá: tỷ đồng).

### Bài B1. Khám phá dữ liệu giá nhà

Đọc dữ liệu vào biến `nha`. Cho biết thống kê mô tả, giá trung bình theo quận (sắp xếp giảm dần),
và vẽ biểu đồ phân tán diện tích – giá, tô màu theo quận.

In [ ]:
# Viết code của bạn ở đây


### Bài B2. Mã hóa biến phân loại và chia dữ liệu

Cột `quan` là chữ, mô hình không dùng trực tiếp được. Dùng `pd.get_dummies(nha, columns=["quan"], drop_first=True)`
để tạo các cột 0/1. Tạo `X` (tất cả cột trừ giá), `y` (giá), chia 80/20 với `random_state=42`.

<details><summary>💡 Gợi ý</summary>

`drop_first=True` bỏ một quận làm nhóm tham chiếu, tránh trùng lặp thông tin.

</details>

In [ ]:
# Viết code của bạn ở đây


### Bài B3. Huấn luyện và so sánh hai mô hình

Huấn luyện Linear Regression và Random Forest Regressor (300 cây). Lập bảng MAE, RMSE, R² trên tập kiểm tra
(dùng lại hàm `danh_gia`). Mô hình nào tốt hơn? MAE có ý nghĩa gì bằng tiền?

In [ ]:
# Viết code của bạn ở đây


### Bài B4. Diễn giải hệ số

In hệ số của mô hình Linear Regression dưới dạng bảng. Trả lời: (a) mỗi m² thêm làm giá tăng bao nhiêu?
(b) nhà cũ thêm 1 năm giảm bao nhiêu? (c) so với quận tham chiếu, quận nào đắt hơn nhiều nhất?

In [ ]:
# Viết code của bạn ở đây


### Bài B5. Định giá căn hộ mới

Dùng mô hình tốt hơn để định giá 3 căn: (1) Quận 7, 75 m², 2 phòng ngủ, 5 năm tuổi, cách trung tâm 8 km;
(2) Thủ Đức, 90 m², 3 phòng ngủ, 2 năm, 12 km; (3) Quận 3, 50 m², 1 phòng ngủ, 15 năm, 3 km.

<details><summary>💡 Gợi ý</summary>

Sau `get_dummies`, dùng `.reindex(columns=Xn.columns, fill_value=0)` để có đúng các cột như lúc huấn luyện.

</details>

In [ ]:
# Viết code của bạn ở đây


## Phần C. Bài tập làm thêm

### Bài C1. Kiểm định chéo (cross-validation)

Một lần chia train/test có thể may rủi. Dùng `cross_val_score(model, X, y, cv=5, scoring="r2")` để đánh giá cả hai mô hình
giá nhà trên 5 lần chia. So sánh trung bình và độ lệch chuẩn của R².

In [ ]:
# Viết code của bạn ở đây


### Bài C2. Pipeline – gom tiền xử lý và mô hình

Dùng `ColumnTransformer` (OneHotEncoder cho `quan`, giữ nguyên các cột số) và `Pipeline` để có một mô hình nhận trực tiếp
dữ liệu thô (cột `quan` dạng chữ). Huấn luyện trên `nha`, rồi định giá trực tiếp DataFrame `moi` ở B5 (chỉ các cột đầu vào).

In [ ]:
# Viết code của bạn ở đây


### Bài C3. Phân tích phần dư

Với mô hình Linear Regression giá nhà, tính phần dư = thực tế − dự báo trên tập kiểm tra. Vẽ histogram phần dư và
biểu đồ phần dư theo giá dự báo. Căn nào bị định giá thấp nhất (phần dư dương lớn nhất)? Đó có thể là "căn đắt bất thường"
hay dữ liệu lỗi?

In [ ]:
# Viết code của bạn ở đây
